In [2]:
import pandas as pd
import numpy as np
import torch
from gensim.models import KeyedVectors

# ========================
# 1. Word2Vecモデルの読み込み
# ========================
print("Loading Word2Vec model...")
word2vec_path = 'C:/Users/eriya/Documents/classcontents/25春夏/100knock2025/chapter06/GoogleNews-vectors-negative300.bin.gz'
model = KeyedVectors.load_word2vec_format(word2vec_path, binary=True)
print("Word2Vec loaded.")

# ========================
# 2. トークンID辞書と埋め込み行列の構築
# ========================
embedding_dim = model.vector_size
max_vocab_size = 50000  # 必要なら調整

token_to_id = {'<PAD>': 0}
id_to_token = {0: '<PAD>'}
embedding_matrix = [np.zeros(embedding_dim)]

print("Building vocab...")
for i, word in enumerate(model.key_to_index.keys()):
    if i >= max_vocab_size:
        break
    vec = model[word]
    token_to_id[word] = i + 1
    id_to_token[i + 1] = word
    embedding_matrix.append(vec)
embedding_matrix = np.array(embedding_matrix)
print("Vocab built. Vocab size:", len(token_to_id))

# ========================
# 3. データの読み込みと変換関数
# ========================
def convert_dataset(filepath):
    df = pd.read_csv(filepath, sep='\t')
    result = []

    for _, row in df.iterrows():
        text = row['sentence']
        label_value = float(row['label'])
        tokens = text.split()
        ids = [token_to_id[token] for token in tokens if token in token_to_id]

        if not ids:
            continue

        result.append({
            'text': text,
            'label': torch.tensor([label_value], dtype=torch.float),
            'input_ids': torch.tensor(ids, dtype=torch.long)
        })

    return result

# ========================
# 4. train/dev データの変換
# ========================
print("Converting train/dev datasets...")
train_data = convert_dataset('train.tsv')
dev_data = convert_dataset('dev.tsv')
print(f"Train examples: {len(train_data)}")
print(f"Dev examples: {len(dev_data)}")

# ========================
# 5. 結果表示（例）
# ========================
print("\n[Example - Train]")
print(train_data[0])

print("\n[Example - Dev]")
print(dev_data[0])


Loading Word2Vec model...
Word2Vec loaded.
Building vocab...
Vocab built. Vocab size: 50001
Converting train/dev datasets...
Train examples: 65018
Dev examples: 872

[Example - Train]
{'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([ 5785,    66,    18,    12, 15095,  1594])}

[Example - Dev]
{'text': "it 's a charming and often affecting journey . ", 'label': tensor([1.]), 'input_ids': tensor([   16, 13259,   640,  5199,  3900])}


In [3]:
import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
import pickle

# 70で出力したファイルをロード
embedding_matrix = np.load('70_embeddings.npy')
with open('70_token_maps.pkl', 'rb') as f:
    maps = pickle.load(f)
    token2id = maps['token2id']
    id2token = maps['id2token']

# SST 読み込み関数定義
def load_sst(path, token2id):
    df = pd.read_csv(path, sep='\t', header=0, usecols=['sentence','label'])
    df['label'] = pd.to_numeric(df['label'], errors='coerce')
    df = df.dropna(subset=['label'])
    examples = []
    for _, row in df.iterrows():
        words = row['sentence'].split()
        ids = [token2id[w] for w in words if w in token2id]
        if not ids:
            continue
        examples.append({'text': row['sentence'],
                         'label': torch.tensor([row['label']], dtype=torch.float),
                         'input_ids': torch.tensor(ids, dtype=torch.long)})
    return examples

train_data = load_sst('train.tsv', token2id)
dev_data   = load_sst('dev.tsv', token2id)

# 必要なら保存
import pickle
with open('71_sst_data.pkl', 'wb') as f:
    pickle.dump({'train': train_data, 'dev': dev_data}, f)
